# 3. Train_Crisis predictions

The crisis run: train on 2000-2007 and predict the held-out 2008-2012 crash into `Data/Predictions_crisis/`.

In [ ]:
# the crisis run: train on 2000-2007, predict the 2008-2012 crash
# Same base as the main Train setup, but ONE training start (2000) and NO global cutoff: each crisis notebook
# passes its own cutoff to the driver (2008 for Train_Crisis, 2006 for Validation_Crisis) and sets its own
# output folder in the run cell. Run this first, then the cells below.
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

DATA_DIR = "Data"

GMM_K = 3
MODEKEY = {("daily", "single"): "", ("weekly", "together"): "wk_", ("weekly", "split"): "split_wk_",
           ("monthly", "together"): "mo_", ("monthly", "split"): "split_mo_"}
MODES = [("daily", "single"), ("weekly", "together"), ("weekly", "split"),
         ("monthly", "together"), ("monthly", "split")]
ALGOS = ["lstm", "xgb", "knn"]
SCALE_FUSIONS = [("daily", ["single"]), ("weekly", ["together", "split"]), ("monthly", ["together", "split"])]
TIERS = ["bare", "gmm", "pooled"]
STARTS = ["2000-01-01"]          # crisis: single training start (2010/2015 are after the cutoffs)
XDAYS = [10, 20, 30, 100]
print("crisis setup ready | STARTS =", STARTS, "| cutoff is per-sheet (2006 / 2008), output folder set in the run cell")

In [ ]:
# read the features table (two parquet parts)
FEATURES = ["logret1", "vol20", "logret20", "range20", "dd60", "hl_range", "close_chg", "open_chg"]
INPUT_COLS = FEATURES + ["Close", "Open"]

def read_features(parts):
    long = pd.concat([pd.read_parquet(p) for p in parts]).set_index("Date")
    return {t: g.drop(columns="ticker") for t, g in long.groupby("ticker")}

feats = read_features([f"{DATA_DIR}/features_1.parquet", f"{DATA_DIR}/features_2.parquet"])
print(len(feats), "tickers")

In [ ]:
# Windows - anchors + builder (+ shared prep)
# Windows + the two prep helpers the runners AND the inspects share, so what you inspect is exactly
# what the grid runs: fit_scaler (train-only StandardScaler) and windows_for (scale -> windows + masks).
def week_starts(dates):
    iso = dates.isocalendar()
    wid = iso["year"].to_numpy() * 100 + iso["week"].to_numpy()
    return np.where(np.r_[True, wid[1:] != wid[:-1]])[0]

def month_starts(dates):
    ym = dates.year.to_numpy() * 100 + dates.month.to_numpy()
    return np.where(np.r_[True, ym[1:] != ym[:-1]])[0]

SCALES = {"daily": None, "weekly": week_starts, "monthly": month_starts}

def default_y(scale, fusion):
    if scale == "daily": return 0
    return 6 if (scale == "monthly" and fusion == "together") else 5

def make_windows(Z, close, dates, x_days, scale, fusion, y):
    day, anc, one, yy, at, nxt = [], [], [], [], [], []
    if scale == "daily":
        for t in range(x_days - 1, len(dates) - 1):
            one.append(Z[t - x_days + 1:t + 1])
            yy.append(int(close[t + 1] > close[t])); at.append(t); nxt.append(t + 1)
    else:
        A = SCALES[scale](dates)
        for j in range(len(A) - 1):
            t, tn = A[j], A[j + 1]
            s = t - x_days + 1
            if s < 0:
                continue
            if fusion == "split":
                if j < y - 1:
                    continue
                day.append(Z[s:t + 1]); anc.append(Z[A[j - y + 1:j + 1]])
            else:
                k = np.searchsorted(A, s)
                if k < y:
                    continue
                one.append(np.vstack([Z[A[k - y:k]], Z[s:t + 1]]))
            yy.append(int(close[tn] > close[t])); at.append(t); nxt.append(tn)
    yy, at, nxt = np.array(yy, "float32"), np.array(at), np.array(nxt)
    X = [np.array(day, "float32"), np.array(anc, "float32")] if fusion == "split" else [np.array(one, "float32")]
    return X, yy, at, nxt

def fit_scaler(df, train_start, cutoff):
    dates = df.index
    m = (dates >= pd.Timestamp(train_start)) & (dates < pd.Timestamp(cutoff))
    return StandardScaler().fit(df.loc[m, INPUT_COLS])

def windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y):
    dates, close = df.index, df["Close"].to_numpy()
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    en = pd.Timestamp(end) if end is not None else dates.max()
    Z = scaler.transform(df[INPUT_COLS])
    X, y_all, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
    tr = (dates[at] >= ts) & (dates[nxt] < co)
    te = (dates[at] >= co) & (dates[at] <= en)
    return Z, X, y_all, at, nxt, tr, te

def frame(df, dates, at, y, prob):
    out = df.loc[dates[at], ["Open", "High", "Low", "Close"]].copy()
    out["y_up"], out["prob_up"] = y, prob
    return out

def flat(X):
    return X.reshape(len(X), -1)

_POOL = {}

In [ ]:
# Regime - fit the GMM (regime / GMM rank)
# fit_gmm fits the GMM on the train features; regime_at re-derives the regime for a config's test rows
# (scaler + windows + GMM, no model training) so the +GMM tier can reuse the bare prob_up.
def fit_gmm(Z, dates, train_start, cutoff, k, seed):
    m = (dates >= pd.Timestamp(train_start)) & (dates < pd.Timestamp(cutoff))
    return GaussianMixture(k, covariance_type="full", n_init=5, random_state=seed).fit(Z[m])

def regime_at(ticker, feats, train_start, cutoff, end, scale, fusion, x_days, gmm_k, seed=42):
    df = feats[ticker].sort_index()
    scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, default_y(scale, fusion))
    gm = fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    return gm.predict(Z[at[te]])

In [ ]:
# LSTM - network + fit
# LSTM network + fit (in memory). Nothing is saved to disk - only the predictions go to parquet.
def build_lstm(steps, n_feat, u1=256, u2=128, dropout=0.01, lr=1e-3):
    m = Sequential([Input((steps, n_feat)),
                    LSTM(u1, return_sequences=True), Dropout(dropout),
                    LSTM(u2), Dropout(dropout),
                    Dense(1, activation="sigmoid")])
    m.compile(optimizer=Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

def steps_of(x_days, scale, fusion, y, branch):
    if scale == "daily":  return x_days
    if fusion == "split": return x_days if branch == 0 else y
    return x_days + y

def _cb():
    return [EarlyStopping("val_accuracy", mode="max", patience=25, restore_best_weights=True),
            ReduceLROnPlateau("val_loss", factor=0.5, patience=5, min_lr=1e-6)]

def fit_lstm(X, y, steps, x_days, scale, epochs=200, batch=16, val_frac=0.1):
    cut = int(len(X) * (1 - val_frac))
    purge = (x_days - 1) if scale == "daily" else x_days // 5
    m = build_lstm(steps, X.shape[2])
    m.fit(X[:cut - purge], y[:cut - purge], validation_data=(X[cut:], y[cut:]),
          epochs=epochs, batch_size=batch, verbose=0, callbacks=_cb())
    return m

def predict_lstm(models, Xlist):
    return np.mean([m.predict(a, verbose=0).ravel() for m, a in zip(models, Xlist)], axis=0)

In [ ]:
# LSTM - pooled (one model over 40)
# Pooled LSTM - one model over all 40 tickers; memoised per (scale, fusion, start, x_days).
def pooled_lstm(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, val_frac=0.1):
    key = ("lstm", scale, fusion, train_start, cutoff, x_days, gmm_k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    np.random.seed(seed); tf.random.set_seed(seed)
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    purge = (x_days - 1) if scale == "daily" else x_days // 5
    Xf, Xv = [[] for _ in range(n)], [[] for _ in range(n)]
    yf, yv = [], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        cut = int(m.sum() * (1 - val_frac))
        if cut - purge < 1 or m.sum() - cut < 1:
            continue
        for i in range(n):
            Xi = X[i][m]; Xf[i].append(Xi[:cut - purge]); Xv[i].append(Xi[cut:])
        yy_m = yy[m]; yf.append(yy_m[:cut - purge]); yv.append(yy_m[cut:])
    yf, yv = np.concatenate(yf), np.concatenate(yv)
    models = []
    for i in range(n):
        m = build_lstm(steps_of(x_days, scale, fusion, y, i), np.vstack(Xf[i]).shape[2])
        m.fit(np.vstack(Xf[i]), yf, validation_data=(np.vstack(Xv[i]), yv),
              epochs=200, batch_size=16, verbose=0, callbacks=_cb())
        models.append(m)
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# LSTM - run_lstm
# run_lstm - bare / +GMM / pooled, any of the five modes. Shares fit_scaler + windows_for with inspect.
def run_lstm(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
             x_days=10, y=None, gmm=False, pooled=False, seed=42, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    np.random.seed(seed); tf.random.set_seed(seed)
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_lstm(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [fit_lstm(X[i][tr], y_all[tr], steps_of(x_days, scale, fusion, y, i), x_days, scale)
                  for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_lstm(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_lstm(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# XGBoost - core + pooled
# XGBoost core - params, tabular prediction, pooled trainer (memoised).
XGB_P = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=5,
             eval_metric="logloss", tree_method="hist", n_jobs=-1)

def predict_tab(models, Xlist):
    return np.mean([m.predict_proba(flat(a))[:, 1] for m, a in zip(models, Xlist)], axis=0)

def pooled_xgb(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k):
    key = ("xgb", scale, fusion, train_start, cutoff, x_days, gmm_k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    Xf, yf = [[] for _ in range(n)], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        if m.sum() == 0:
            continue
        for i in range(n): Xf[i].append(flat(X[i][m]))
        yf.append(yy[m])
    Xf = [np.vstack(a) for a in Xf]; yf = np.concatenate(yf)
    models = [XGBClassifier(random_state=seed, **XGB_P).fit(Xf[i], yf.astype(int)) for i in range(n)]
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# XGBoost - run_xgb
# run_xgb.
def run_xgb(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
            x_days=10, y=None, gmm=False, pooled=False, seed=42, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    np.random.seed(seed)
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_xgb(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [XGBClassifier(random_state=seed, **XGB_P).fit(flat(X[i][tr]), y_all[tr].astype(int)) for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_tab(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_tab(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# k-NN - core + pooled
# k-NN core (k=50) - pooled index (memoised).
def pooled_knn(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, k=50):
    key = ("knn", scale, fusion, train_start, cutoff, x_days, gmm_k, k)
    if _POOL.get("k") == key:
        return _POOL["v"]
    ts, co = pd.Timestamp(train_start), pd.Timestamp(cutoff)
    rows = [feats[t].sort_index().loc[(feats[t].index >= ts) & (feats[t].index < co), INPUT_COLS] for t in sorted(feats)]
    scaler = StandardScaler().fit(pd.concat(rows))
    gm = GaussianMixture(gmm_k, covariance_type="full", n_init=5, random_state=seed).fit(scaler.transform(pd.concat(rows)))
    n = 2 if fusion == "split" else 1
    Xf, yf = [[] for _ in range(n)], []
    for t in sorted(feats):
        df = feats[t].sort_index(); dates, close = df.index, df["Close"].to_numpy()
        Z = scaler.transform(df[INPUT_COLS]); X, yy, at, nxt = make_windows(Z, close, dates, x_days, scale, fusion, y)
        m = (dates[at] >= ts) & (dates[nxt] < co)
        if m.sum() == 0:
            continue
        for i in range(n): Xf[i].append(flat(X[i][m]))
        yf.append(yy[m])
    Xf = [np.vstack(a) for a in Xf]; yf = np.concatenate(yf)
    models = [KNeighborsClassifier(k).fit(Xf[i], yf.astype(int)) for i in range(n)]
    _POOL.clear(); _POOL.update(k=key, v=(scaler, gm, models))
    return scaler, gm, models

In [ ]:
# k-NN - run_knn
# run_knn.
def run_knn(ticker, feats, train_start, cutoff, end=None, scale="daily", fusion="single",
            x_days=10, y=None, gmm=False, pooled=False, seed=42, k=50, gmm_k=3):
    y = default_y(scale, fusion) if y is None else y
    df = feats[ticker].sort_index()
    if pooled:
        scaler, gm, models = pooled_knn(feats, train_start, cutoff, end, scale, fusion, x_days, y, seed, gmm_k, k)
    else:
        scaler = fit_scaler(df, train_start, cutoff)
    Z, X, y_all, at, nxt, tr, te = windows_for(df, scaler, train_start, cutoff, end, scale, fusion, x_days, y)
    if not pooled:
        models = [KNeighborsClassifier(k).fit(flat(X[i][tr]), y_all[tr].astype(int)) for i in range(len(X))]
    res    = frame(df, df.index, at[te], y_all[te], predict_tab(models, [a[te] for a in X]))
    res_tr = frame(df, df.index, at[tr], y_all[tr], predict_tab(models, [a[tr] for a in X]))
    if not (gmm or pooled):
        return res, res_tr
    gm_model = gm if pooled else fit_gmm(Z, df.index, train_start, cutoff, gmm_k, seed)
    res["regime"] = gm_model.predict(Z[at[te]])
    res_tr["regime"] = gm_model.predict(Z[at[tr]])
    return res, res_tr

In [ ]:
# Crisis driver - the Validation pipeline generalized to any (cutoff, window, folder)
# Saves + prints progress PER TICKER (like the Validation -> parquets notebook), so a crash loses at most
# one ticker. Only the cutoff, the predicted end and the output folder are arguments; trains only CTICKERS
# (tickers whose history reaches back to 2000). Reuses run_lstm / run_xgb / run_knn, regime_at and the knobs above.
RUNNERS = {"lstm": run_lstm, "xgb": run_xgb, "knn": run_knn}
# only train tickers whose history reaches back to 2000 (the crisis start); drop later IPOs
CTICKERS = [tk for tk in sorted(feats) if feats[tk].index.min().year <= 2000]
_drop = [(tk, str(feats[tk].index.min().date())) for tk in sorted(feats) if tk not in CTICKERS]
print(f"crisis universe: {len(CTICKERS)}/{len(feats)} tickers have data from 2000 | dropped {len(_drop)}: {_drop}")

def train_tier_win(algo, scale, fusion, tier, cutoff, end, outdir):
    modek = MODEKEY[(scale, fusion)]
    path = f"{outdir}/{tier}_{modek}{algo}_preds.parquet"
    have = pd.read_parquet(path) if os.path.exists(path) else pd.DataFrame()
    done = set(zip(have["ticker"], have["train_start"].astype(str).str[:10], have["x_days"].astype(int))) if len(have) else set()
    bpath = f"{outdir}/bare_{modek}{algo}_preds.parquet"
    bare = pd.read_parquet(bpath) if (tier == "gmm" and os.path.exists(bpath)) else None
    parts, n_new = ([have] if len(have) else []), 0
    grid = [(xd, st, tk) for xd in XDAYS for st in STARTS for tk in CTICKERS]
    total = len(grid)
    n_done = sum((tk, st, int(xd)) in done for xd, st, tk in grid)
    print(f"    already trained {n_done}/{total}")
    for xd, st, tk in grid:
        if (tk, st, int(xd)) in done:
            continue
        if tier == "gmm" and bare is not None:
            m = (bare["ticker"] == tk) & (bare["train_start"].astype(str).str[:10] == st) & (bare["x_days"].astype(int) == int(xd))
            if m.any():
                out = bare.loc[m, ["ticker", "y_up", "prob_up", "train_start", "x_days", "scale", "fusion"]].reset_index(drop=True)
                out.insert(3, "regime", regime_at(tk, feats, st, cutoff, end, scale, fusion, xd, GMM_K))
                out["tier"] = "gmm"
                parts.append(out); n_new += 1; n_done += 1
                pd.concat(parts, ignore_index=True).to_parquet(path)
                print(f"    {n_done:3d}/{total} regime  {tk:5s} start={st[:4]} x={xd}")
                continue
        kw = dict(scale=scale, fusion=fusion, x_days=xd, gmm_k=GMM_K)
        if tier == "gmm":      kw["gmm"] = True
        elif tier == "pooled": kw["pooled"] = True
        res, _ = RUNNERS[algo](tk, feats, st, cutoff, end=end, **kw)
        keep = ["y_up", "prob_up"] + (["regime"] if tier != "bare" else [])
        out = res[keep].reset_index(drop=True)
        out.insert(0, "ticker", tk)
        out["train_start"], out["x_days"] = st, int(xd)
        out["scale"], out["fusion"], out["tier"] = scale, fusion, tier
        parts.append(out); n_new += 1; n_done += 1
        pd.concat(parts, ignore_index=True).to_parquet(path)
        print(f"    {n_done:3d}/{total} trained {tk:5s} start={st[:4]} x={xd}")
    return n_new

def run_matrix_win(cutoff, end, outdir, label):
    os.makedirs(outdir, exist_ok=True)
    print(f"=== {label}: train < {cutoff}, predict [{cutoff} .. {end}] -> {outdir} ===")
    tiers = [(a, s, f, t) for a in ALGOS for s, fus in SCALE_FUSIONS for f in fus for t in TIERS]
    for i, (a, s, f, t) in enumerate(tiers, 1):
        print(f"[{i:2d}/{len(tiers)}] {a:4s} {s}/{f} {t:6s} - training ...")
        n = train_tier_win(a, s, f, t, cutoff, end, outdir)
        print(f"[{i:2d}/{len(tiers)}] {a:4s} {s}/{f} {t:6s}: +{n} trained")
    print("done")

def stack_universal(outdir):
    frames = []
    for algo in ALGOS:
        for scale, fus in SCALE_FUSIONS:
            for fusion in fus:
                for tier in TIERS:
                    path = f"{outdir}/{tier}_{MODEKEY[(scale, fusion)]}{algo}_preds.parquet"
                    if os.path.exists(path):
                        frames.append(pd.read_parquet(path).assign(algo=algo))
    allp = pd.concat(frames, ignore_index=True)
    allp.to_parquet(f"{outdir}/all_predictions.parquet")
    print("universal:", allp.shape, "->", f"{outdir}/all_predictions.parquet")
    return allp

In [ ]:
# Train_Crisis - train 2000-2007, predict OUT-OF-SAMPLE the 2008-2012 crisis (held out = df_validation)
# Only the cutoff (2008) + output folder differ from the main Train notebook. Heavy + resumable (re-run to continue).
TDIR = f"{DATA_DIR}/Predictions_crisis"
run_matrix_win("2008-01-01", "2012-12-31", TDIR, "Train_Crisis 2008-2012 | model trains 2000-2007")
T = stack_universal(TDIR)
print("-> Data analysis: VALID_PATH =", f"{TDIR}/all_predictions.parquet",
      "| WIN['valid'] = ('2008-01-01','2012-12-31')")